# Functional Programming in Python (8/21/26 – 9/23/26)

---

*Codecademy — MLE Path: SWE for ML → **Functional Programming**. Concise grad-student notes. Companion to [OOP Part 1](OOP_Python_Lesson.ipynb) and [OOP Part 2](OOP_Python_Part2_Lesson.ipynb).*

> **Scope:** immutable data and `namedtuple`; `lambda`; the three higher-order functions `filter` / `map` / `reduce` and every way of nesting them; then applying the same chains to real data — streaming a CSV through `csv.reader` and loading JSON with `json.load`.


## Functional Programming

Ten sublessons, in three movements: the building blocks (immutability, lambdas, the three functions), the four ways to combine them, and the data-file payoff (CSV, JSON).


### Immutable Data Types

**Functional programming** favors **pure functions** and evaluating **expressions** rather than executing statements. **Immutable** data types are central to it, because they give you:

- thread-safe data manipulation — nothing can mutate underneath a concurrent reader
- protection against accidentally changing a value

A tuple of tuples holds mixed records fine:

```
students = (("Scott", 28, 'A'), ("Nicole", 26, 'B'), ("John", 29, 'D'))
```

but it's error-prone: you have to remember that position 1 is the age and position 2 is the grade. `namedtuple` from `collections` fixes that — same immutability, but fields have names.


In [1]:
from collections import namedtuple

# Create a tuple subclass called student
student = namedtuple("student", ["name", "age", "grade"])

# Create tuples for the three students
scott = student("Scott", 28, 'A')
nicole = student("Nicole", 26, 'B')
john = student("John", 29, 'D')

# Access Scott's information by field name, not position
print(scott.name)   # -> Scott
print(scott.age)    # -> 28
print(scott.grade)  # -> A

# Package them into one tuple
students = (scott, nicole, john)
print(students[1].name)  # -> Nicole


Scott
28
A
Nicole


`namedtuple("student", [...])` returns a *tuple subclass*; calling it builds an instance. Convention (not a requirement) is to give the subclass and the variable holding it the same name, as with `student` above.


**Exercise** — same pattern on a different record: a `country` namedtuple with `name`, `capital`, `continent`, three instances, packed into one tuple.


In [2]:
from collections import namedtuple

# Checkpoint 1 code goes here.
country = namedtuple("country", ["name", "capital", "continent"])
# Checkpoint 2 code goes here.
France = country("France", "Paris", "Europe")
Japan = country("Japan", "Tokyo", "Asia")
Senegal = country("Senegal", "Dakar", "Africa")

# Checkpoint 3 code goes here.
countries = (France, Japan, Senegal)


No output — the checkpoint only builds the data. `countries[2].capital` would give `Dakar`.


### Review of Lambda Functions

A **lambda** is a short **anonymous** function: it can take several parameters but returns exactly one value. Lambdas matter in functional programming because they let you pass a function as an argument without ceremony — either stored in a variable or written inline at the call site.

Motivating example: price a sheet of metal as *price per square meter × area*, where the sheet can be cut as a rectangle or a triangle. `total_cost` takes the price per square meter, a tuple of dimensions, and **a function that computes the area** — so the shape is a parameter, not a branch.

- rectangle: $\text{length} \times \text{width}$
- triangle: $0.5 \times (\text{base} \times \text{height})$

First without lambdas:


In [3]:
def rect(l, w):
  return l * w

def tri(b, h):
  return 0.5 * (b * h)

# ppsm: price per square meter
# dim: dimensions tuple
def total_cost(ppsm, dim, area):
  return ppsm * area(dim[0], dim[1])

# Rectangular sheet with length of 5 and width of 5
print(total_cost(3, (5, 5), rect))  # -> 75

# Triangular sheet with base of 6 and height of 7
print(total_cost(4, (6, 7), tri))   # -> 84.0


75
84.0


The two named definitions exist only to be handed to `total_cost`, so replace them with lambdas written inline:


In [4]:
# ppsm: price per square meter
# dim: dimensions tuple
def total_cost(ppsm, dim, area):
  return ppsm * area(dim[0], dim[1])

print(total_cost(3, (5, 5), lambda l, w: l*w))       # -> 75
print(total_cost(4, (6, 7), lambda b, h: 0.5 * b*h)) # -> 84.0

# Lambdas can also be stored in a variable
rect = lambda l, w: l * w
tri = lambda b, h: 0.5 * b * h
print(rect(5, 5), tri(6, 7))  # -> 25 21.0


75
84.0
25 21.0


The catch: lambdas are only for **short and fast** functions. A single expression, no statements — anything long or branching has to stay a `def`.


**Exercise** — `odd_or_even(n, even_function, odd_function)` dispatches on `n % 2`, calling whichever function fits. Then convert `squared`/`cubed` into lambdas and wire them in so odd numbers get **squared** and even numbers get **cubed** — which means passing `cube` as the *even* argument and `square` as the *odd* one.


In [5]:
""" 
def squared(x):
  return x * x

def cubed(x):
  return x*x*x
"""
def odd_or_even(n, even_function, odd_function):
  if n % 2 == 0: 
    return even_function(n)
  else: 
    return odd_function(n)


# Checkpoint 2 code goes here.
square = lambda x: x**2
cube = lambda x: x**3

# Checkpoint 3 code goes here.
test = odd_or_even(5,  cube, square)

print(test) # Uncomment the print function to see the results of Checkpoint 3.
# -> 25


25


`5` is odd, so `odd_function` runs — that's `square` — giving `25`. The argument *order* is what encodes the rule here, not the lambda bodies.


### Review of `filter()`, `map()`, and `reduce()`

Three **higher-order functions** built into Python: each takes an iterable plus a processing function and hands back another iterable (or, for `reduce`, a single value). They turn **imperative** loops — build a mutable list, append as you go — into **declarative** expressions that say *what* you want rather than *how* to accumulate it.

| Function | Processing function | Result |
| --- | --- | --- |
| `filter(pred, it)` | a **predicate** — one arg, returns a bool | the elements where the predicate is true |
| `map(f, it)` | one arg, returns the transformed value | every element, transformed |
| `reduce(f, it)` | an **accumulator** — exactly two args | one collapsed value |

#### `filter()`

Imperative — a mutable `ret` list and an explicit loop:


In [6]:
nums = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)

# filter_values is not a higher-order function
def filter_values(predicate, lst):

  # Mutable list required because this example is imperative, not declarative
  ret = []
  for i in lst:
    if predicate(i):
      ret.append(i)
  return ret

filtered_numbers = filter_values(lambda x: x % 2 == 0, nums)

print(filtered_numbers)
# -> [2, 4, 6, 8, 10]


[2, 4, 6, 8, 10]


Declarative — the same thing in one line:


In [7]:
nums = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)

filtered_numbers = filter(lambda x: x % 2 == 0, nums)

print(tuple(filtered_numbers))
# -> (2, 4, 6, 8, 10)


(2, 4, 6, 8, 10)


#### `map()`

Same shift, but applying a function to *every* element instead of keeping some. Imperative first:


In [8]:
nums = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)

def mapper(function, lst):
  ret = []
  for i in lst:
    ret.append(function(i))
  return ret

mapped_numbers  = mapper(lambda x: x*x, nums)

print(tuple(mapped_numbers))
# -> (1, 4, 9, 16, 25, 36, 49, 64, 81, 100)


(1, 4, 9, 16, 25, 36, 49, 64, 81, 100)


In [9]:
numbers = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)

mapped_numbers = map(lambda x: x*x, numbers)

print(tuple(mapped_numbers))
# -> (1, 4, 9, 16, 25, 36, 49, 64, 81, 100)


(1, 4, 9, 16, 25, 36, 49, 64, 81, 100)


#### `reduce()`

`reduce()` takes an iterable and a two-parameter function called an **accumulator**, and works through the iterable collapsing it down to a single value. Summing a tuple, imperatively:


In [10]:
nums = (2, 6, 7, 9, 1, 4, 8)

sum = 0

for i in nums:
  sum += i

print(sum)  # -> 37


37


Declaratively, `x` is the running total and `y` is the next element:


In [11]:
"""
In Python 3, the reduce() function has been moved to the functools library, so we need to import it before we can use it.
"""
from functools import reduce

nums = (2, 6, 7, 9, 1, 4, 8)

reduced_nums = reduce(lambda x, y: x + y, nums) # reduced_nums is a number

print(reduced_nums)  # -> 37


37


Two things worth remembering:

- The `reduce()` lambda takes **exactly two** parameters — no more, no fewer.
- `filter()` and `map()` return lazy iterators, not tuples, which is why every example wraps the result in `tuple(...)` to print it. Iterate one twice and the second pass is empty.

The payoff is shorter, cleaner code. The next sublessons combine these three functions — e.g. `reduce` over a `map` over a `filter` — rather than using them one at a time.


**Exercise** — three checkpoints, each applying one function to the same starter tuple `nums`:

1. `filter()` + lambda to drop the odd numbers (`x % 2 == 0` keeps evens).
2. `map()` + lambda to multiply every element by 3.
3. `reduce()` + lambda to sum the whole tuple.

The `filter`/`map` results are wrapped in `tuple(...)` to print; `reduce` already yields a plain number.


In [12]:
from functools import reduce
nums = (16, 2, 19, 22, 10, 23, 16, 2, 27, 29, 19, 26, 12, 20, 16, 29, 6, 2, 12, 20)

# Checkpoint 1 code goes here.
filtered_numbers = filter(lambda x: x % 2 == 0, nums)
print(tuple(filtered_numbers))
# -> (16, 2, 22, 10, 16, 2, 26, 12, 20, 16, 6, 2, 12, 20)

# Checkpoint 2 code goes here.
mapped_numbers = map(lambda x: 3*x, nums)
print(tuple(mapped_numbers))
# -> (48, 6, 57, 66, 30, 69, 48, 6, 81, 87, 57, 78, 36, 60, 48, 87, 18, 6, 36, 60)

# Checkpoint 3 code goes here.
sums = reduce(lambda x, y: x + y, nums)
print(sums)
# -> 328


(16, 2, 22, 10, 16, 2, 26, 12, 20, 16, 6, 2, 12, 20)
(48, 6, 57, 66, 30, 69, 48, 6, 81, 87, 57, 78, 36, 60, 48, 87, 18, 6, 36, 60)
328


Note the result *shapes*: checkpoint 1 has fewer elements than `nums` (14 of 20), checkpoint 2 has the same 20, and checkpoint 3 is a single number — map keeps the count, filter shrinks it, reduce collapses it. Naming the sum `sums` rather than `sum` also avoids shadowing the built-in `sum()`.


### Mapping a Filtered Collection

The first combination. If the sentence in your head is *"I need to map only the values that have property x,"* that is `map` wrapped around `filter`:

```python
map(mapping_function, filter(filtering_function, iterable))
```

The inner `filter` runs first and decides *which* elements survive; the outer `map` then decides *what each survivor becomes*. Read it inside-out.

**Example.** Records for students in 100-level math classes as a `student(name, grade, course_number)` namedtuple. Anyone with a B or better gets registered for Math 201 with a placeholder grade `'X'` (not finished yet). So: *filter* to grade B-or-better, then *map* each survivor to a new record with `'X'` and `201`.


In [13]:
from collections import namedtuple

# Create a tuple subclass called student
student = namedtuple("student", ["name", "grade", "course_number"])

# Create the records for the students in the form of tuples
peter = student("Peter", 'B', 101)
amanda = student("Amanda", 'C', 101 )
sarah = student("Sarah", 'A', 102)
lisa = student("Lisa", 'D', 101)
alex = student("Alex", 'A', 102)
maria = student("Maria", 'B', 101)
andrew = student("Andrew", 'C', 102)

math_class = (peter, amanda, sarah, lisa, alex, maria, andrew)


Now the one-liner. `q.grade <= 'B'` is the filter, `student(s.name, 'X', 201)` is the map:


In [14]:
math_201 = map(lambda s: student(s.name, 'X', 201), filter(lambda q: q.grade <= 'B', math_class))

print(tuple(math_201))
# -> (student(name='Peter', grade='X', course_number=201),
#     student(name='Sarah', grade='X', course_number=201),
#     student(name='Alex', grade='X', course_number=201),
#     student(name='Maria', grade='X', course_number=201))


(student(name='Peter', grade='X', course_number=201), student(name='Sarah', grade='X', course_number=201), student(name='Alex', grade='X', course_number=201), student(name='Maria', grade='X', course_number=201))


Why `<=` for "B or *better*"? Python compares strings by alphabet position, and `'A' < 'B' < 'C'`. So the academically *highest* grade is the *smallest* string, and "B or above" is `grade <= 'B'`. The comparison direction is reversed from what the English suggests — an easy place to slip.

Other things to notice:

- The filter lambda uses `q` and the map lambda uses `s` — two separate functions, each with its own parameter name. Nothing forces them to differ, but it makes clear each lambda only sees its own argument.
- The map builds a **new** `student` rather than editing the old one. Namedtuples are immutable, so this is the only option — and it's the functional style anyway: transform, don't mutate.
- Order is preserved: the four survivors come out in the same order they appeared in `math_class`.
- `map` returns a lazy iterator, so the result is wrapped in `tuple(...)` to print.


**Exercise** — two checkpoints on the same `nums`, both `map` over `filter` with lambdas:

1. Double every value greater than 10 → `greater_than_10_doubled`.
2. Convert this imperative code to declarative → `functional_way`:

```python
lst = []
for i in nums:
  if i % 3 == 0:
    lst.append(i)

for i in range(len(lst)):
  lst[i] = lst[i] * 3

print(tuple(lst))
```

The first loop is the filter (`i % 3 == 0`), the second loop is the map (`* 3`).


In [15]:
nums = (2, 12, 5, 8, 9, 3, 16, 7, 13, 19, 21, 1, 15, 4, 22, 20, 11)

# Checkpoint 1 code goes here.
greater_than_10_doubled = map(lambda m: m*2, filter(lambda y: y > 10, nums))
print(tuple(greater_than_10_doubled))
# -> (24, 32, 26, 38, 42, 30, 44, 40, 22)

# Checkpoint 2 code goes here.
functional_way = map(lambda i: i*3, filter(lambda m: m%3 == 0, nums))
print(tuple(functional_way))
# -> (36, 27, 9, 63, 45)


(24, 32, 26, 38, 42, 30, 44, 40, 22)
(36, 27, 9, 63, 45)


The translation in checkpoint 2 is mechanical: an `if` inside a loop that appends becomes `filter`, a loop that overwrites each slot in place becomes `map`, and the two loops nest as `map(..., filter(...))`. The imperative version also needed a mutable `lst` and an index-based second loop; the declarative one needs neither.


### Reducing a Filtered Collection

Second combination: `reduce` wrapped around `filter`. The sentence is *"filter the collection for x, then collapse the survivors to one value"*:

```python
reduce(accumulator, filter(filtering_function, iterable))
```

**Example.** A restaurant menu as `menu_item(name, dish_type, price)` namedtuples. Find the least expensive appetizer — "filter the menu for appetizers and return the cheapest one." Imperatively, that's a filtering comprehension followed by a running-minimum loop:


In [16]:
from collections import namedtuple
from functools import reduce

# Prices are in USD
menu_item = namedtuple("menu_item", ["name", "dish_type", "price"])

jsp = menu_item("Jumbo Shrimp Platter", "Appetizer", 29.95)
lc = menu_item("Lobster Cake", "Appetizer", 30.95)
scb = menu_item("Sizzling Canadian Bacon", "Appetizer", 9.95)
ccc = menu_item("Codecademy Crab Cake", "Appetizer", 32.95)
cs = menu_item("Caesar Salad", "Salad", 14.95)
mgs = menu_item("Mixed Green Salad", "Salad", 10.95)
cp = menu_item("Codecademy Potatoes", "Side", 34.95)
mp = menu_item("Mashed Potatoes", "Side", 14.95)
a = menu_item("Asparagus", "Side", 15.95)
rs = menu_item("Ribeye Steak", "Entree", 75.95)
phs = menu_item("Porter House Steak", "Entree", 131.95)
grs = menu_item("Grilled Salmon", "Entree", 36.95)

menu = (jsp, lc, scb, ccc, cs, mgs, cp, mp, a, rs, phs, grs)


In [17]:
# apps only contain items with dish_type == "Appetizer"
apps = [i for i in menu if i.dish_type == "Appetizer"]
cheapest_app = apps[0]
for i in apps:
  if i.price < cheapest_app.price:
    cheapest_app = i

print(cheapest_app)
# -> menu_item(name='Sizzling Canadian Bacon', dish_type='Appetizer', price=9.95)


menu_item(name='Sizzling Canadian Bacon', dish_type='Appetizer', price=9.95)


Declaratively, the comprehension becomes `filter` and the running-minimum loop becomes `reduce`:


In [18]:
cheapest_app = reduce(lambda x, y: x if x.price < y.price else y, filter(lambda x: x.dish_type == "Appetizer", menu))

print(cheapest_app)
# -> menu_item(name='Sizzling Canadian Bacon', dish_type='Appetizer', price=9.95)


menu_item(name='Sizzling Canadian Bacon', dish_type='Appetizer', price=9.95)


How to read the accumulator `lambda x, y: x if x.price < y.price else y`: `x` is the cheapest item *seen so far*, `y` is the next candidate, and the lambda returns whichever is cheaper. That winner becomes the new `x` for the next comparison, so after the last element `x` is the overall minimum. This is the same one-at-a-time rhythm as the sum example — just with "keep the smaller" in place of "add".

Two things to notice:

- The accumulator returns a whole `menu_item`, not a price. `reduce` can carry any object as its running value, which is why the result prints as a full record.
- The imperative version had to seed `cheapest_app = apps[0]` by hand; `reduce` does that automatically by starting with the first element. (It would raise on an empty filter result — a menu with no appetizers — where the imperative version would raise on `apps[0]` too.)


**Exercise** — two checkpoints on the same `menu`, both `reduce` over `filter` with lambdas:

1. Most expensive **entree** → `entree`. Same accumulator shape as the example, with `>` instead of `<`.
2. Least expensive dish that is a **side or a salad** → `least_expensive`. The filter predicate is an `or` of two `dish_type` checks.


In [19]:
entree = 0
least_expensive = 0

# Code for Checkpoint 1 goes here.
entree = reduce(lambda x, y: x if x.price > y.price else y, filter(lambda z: z.dish_type == "Entree", menu))
print(entree)
# -> menu_item(name='Porter House Steak', dish_type='Entree', price=131.95)

# Code for Checkpoint 2 goes here.
least_expensive = reduce(lambda x,y: x if x.price < y.price else y, filter(lambda d: d.dish_type == "Side" or d.dish_type == "Salad", menu))
print(least_expensive)
# -> menu_item(name='Mixed Green Salad', dish_type='Salad', price=10.95)


menu_item(name='Porter House Steak', dish_type='Entree', price=131.95)
menu_item(name='Mixed Green Salad', dish_type='Salad', price=10.95)


Flipping `<` to `>` in the accumulator turns "keep the cheaper" into "keep the pricier" — the `reduce` machinery is identical. For checkpoint 2 the filter could also be written `d.dish_type in ("Side", "Salad")`, which scales better than chaining `or`s once there are three or more categories.


### Reducing a Mapped Collection

Third combination: `reduce` wrapped around `map`.

```python
reduce(accumulator, map(mapping_function, iterable))
```

**Why you need the map step.** The accumulator handed to `reduce` must have its two parameters and its return value all of the **same type** — in `lambda x, y: x + y` over numbers, `x`, `y`, and the result are all numbers, and the result feeds back in as the next `x`. A dictionary whose values are `(units_sold, price_per_unit)` tuples can't be reduced straight to a number: the elements aren't numbers. So `map` first converts each entry into a number (its line-item cost), then `reduce` sums those.

**Iterating a dict.** Passing a dictionary as the iterable walks its **keys**, so the mapping lambda receives a key `q` and looks the value up with `costs[q]`.


In [20]:
from functools import reduce

# Dictionary entry: {"name": (number_of_units_sold, price_per_unit)}
costs = {"shirt": (4, 13.00), "shoes":(2, 80.00), "pants":(3, 100.00), "socks":(5, 5.00)}

k = reduce(lambda x, y: x+y, map(lambda q: costs[q][0] * costs[q][1], costs))
print(k)  # -> 537.0


537.0


To see the two stages separately, run just the `map` and wrap it in a tuple — these are the values `reduce` then adds up:


In [21]:
# The map step alone — one line item per key
print(tuple(map(lambda q: costs[q][0] * costs[q][1], costs)))
# -> (52.0, 160.0, 300.0, 25.0)


(52.0, 160.0, 300.0, 25.0)


`costs[q][0]` is the units sold and `costs[q][1]` the unit price, so the map turns each key into `units × price`. After that the `reduce` lambda only ever sees numbers, and the type rule is satisfied.

This dict-to-number pattern — map a lookup over the keys, then reduce — is the one that matters for **JSON** data later in the lesson, since parsed JSON arrives as nested dicts.


**Exercise** — `fruits` maps each fruit to a 3-tuple of units sold on days 1, 2, 3. Find the total sold across all fruits and all days with `map` + `reduce` and lambdas → `total_fruits`. Same shape as the `costs` example: the map turns each key into one number (that fruit's three-day total), the reduce sums those.


In [22]:
from functools import reduce

fruits = {"Grape":(4, 6, 2), "Lemon":(7, 3, 1), "Orange":(5, 8, 1), "Apple":(2, 8, 10), "Watermelon":(0, 9, 6)}

total_fruits = 0

total_fruits = reduce(lambda x, y: x + y, map(lambda q: fruits[q][0] + fruits[q][1] + fruits[q][2], fruits))

print(total_fruits)
# -> 72


72


The map step produces the per-fruit totals (12, 11, 14, 20, 15), and the reduce adds them to 72. The mapping lambda could also be written `lambda q: sum(fruits[q])`, which doesn't change if a fourth day is added — the explicit `[0] + [1] + [2]` would.


### Combining All Three Higher-Order Functions

All three at once: transform the collection, keep some of it, collapse what's left to one value.

```python
reduce(accumulator, filter(predicate, map(transform, iterable)))
```

Read it **inside-out**: `map` runs first, `filter` sees the mapped values, `reduce` sees the filtered ones. (`map` and `filter` can swap order when the predicate is about the *original* values rather than the transformed ones.)

**Example.** Same `costs` inventory as before, extended with ties and a watch. Wanted: the total of the sale values that were **150 or less**.

1. **map** each item to its total sale value, `units × price`.
2. **filter** for sale values ≤ 150.
3. **reduce** the survivors to one sum.


In [23]:
from functools import reduce

# Dictionary entry: {"name": (number_of_units_sold, price_per_unit)}
costs = {"shirt": (4, 13.00), "shoes":(2, 80.00), "pants":(3, 100.00), "socks":(5, 5.00), "ties":(3, 14.00), "watch":(1, 145.00)}

k = reduce(lambda x, y: x + y, filter(lambda r: r <= 150.00, map(lambda q: costs[q][0] * costs[q][1], costs)))

print(k)  # -> 264.0


264.0


The one-liner is dense, so here it is peeled apart one layer at a time. Each stage rebuilds the inner iterators because `map`/`filter` results are single-use:


In [24]:
# Stage 1 — map: total sale value per item
sales = map(lambda q: costs[q][0] * costs[q][1], costs)
print(tuple(sales))
# -> (52.0, 160.0, 300.0, 25.0, 42.0, 145.0)

# Stage 2 — filter: keep sale values of 150 or less
kept = filter(lambda r: r <= 150.00, map(lambda q: costs[q][0] * costs[q][1], costs))
print(tuple(kept))
# -> (52.0, 25.0, 42.0, 145.0)

# Stage 3 — reduce: sum the survivors
print(reduce(lambda x, y: x + y, filter(lambda r: r <= 150.00, map(lambda q: costs[q][0] * costs[q][1], costs))))
# -> 264.0


(52.0, 160.0, 300.0, 25.0, 42.0, 145.0)
(52.0, 25.0, 42.0, 145.0)
264.0


Notice what the filter is testing: `r` is a **sale value** produced by the map, not a dict key. Shoes (160) and pants (300) are dropped because their *totals* exceed 150, even though each unit price is under 150. The watch (145) survives on the same logic. If the question had been "items whose *unit price* is ≤ 150", the filter would have to run on the keys *before* the map — the order of the inner two calls is part of the meaning.

Note also that each lambda gets its own parameter name (`q` for keys, `r` for sale values, `x`/`y` for the running sum) — a small habit that makes the type at each layer obvious.


**Exercise** — two checkpoints, each using all three functions with lambdas:

1. From `costs`, total the sale values that were **greater than** 150 → `total`. The complement of the worked example: map → filter `> 150` → reduce sum.
2. From `nums`, take the numbers **less than 10**, add 5 to each, and multiply them all together → `product`. Here the filter is on the *original* values, so it goes **inside** the map: filter → map → reduce product.


In [25]:
from functools import reduce

costs = {"shirt": (4, 13.00), "shoes":(2, 80.00), "pants":(3, 100.00), "socks":(5, 5.00), "ties":(3, 14.00), "watch":(1, 145.00)}

nums = (24, 6, 7, 16, 8, 2, 3, 11, 21, 20, 22, 23, 19, 12, 1, 4, 17, 9, 25, 15)

# Code for Checkpoint 1 goes here.
total = reduce(lambda x,y: x+y, filter(lambda r: r > 150.00, map(lambda q: costs[q][0]*costs[q][1], costs)))
print(total)
# -> 460.0

# Code for Checkpoint 2 goes here.
product = -1

product = reduce(lambda x, y: x * y, map(lambda z: z + 5, filter(lambda q: q < 10, nums)))

print(product)
# -> 72648576


460.0
72648576


Checkpoint 1 keeps shoes (160) and pants (300) — exactly the two the worked example dropped — so the two answers partition the inventory: 264 + 460 = 724, the full total.

Checkpoint 2 shows the other nesting order. `filter` runs first on `nums` and keeps (6, 7, 8, 2, 3, 1, 4, 9); `map` then adds 5 to give (11, 12, 13, 7, 8, 6, 9, 14); `reduce` multiplies those to 72648576. Testing `q < 10` *after* adding 5 would have kept a different set, so which call sits innermost is decided by whether the condition is about the raw or the transformed values.


### Importing Data From a CSV File

Where this pays off in data science: higher-order functions process data files efficiently. The running example is `zillow.csv` — Zillow housing data for Tallahassee, FL, one house per line, no header:

| Index | Square Footage | Year | List Price (USD) |
| --- | --- | --- | --- |
| 1 | 2222 | 1981 | 250000 |
| 2 | 1628 | 2009 | 185000 |
| 3 | 3824 | 1954 | 399000 |
| 4 | 1137 | 1993 | 150000 |
| 5 | 3560 | 1973 | 315000 |

Three pieces:

1. **Open the file and build a reader.** `csv.reader(...)` returns an **iterator**, not the data. CSVs can run to millions of lines, so loading everything into RAM is impractical — the reader keeps a pointer into the file and only yields the next line when `next(reader)` is called.
2. **A `namedtuple` for each record** — `house(index, square_footage, year, list_price)`.
3. **`map` a lambda over the reader** to turn each line into a `house`. Every field arrives as a **string** regardless of its intended type, so each one is cast with `int(...)`. Field order follows the column order in the file.

`map` calls `next(reader)` for you, so the whole file is consumed lazily and one line at a time. The `map` and any `print` must sit *inside* the `with` block — once the block exits the file is closed and the iterator has nothing to read from.


In [26]:
import csv
from collections import namedtuple

house = namedtuple("house", ["index", "square_footage", "year", "list_price"])

with open('zillow.csv', newline = '') as csvfile:
  reader = csv.reader(csvfile, delimiter=',', quotechar='|')

  # Read record into namedtuple: house(index, square_footage, year, list_price)
  h = map(lambda x: house(int(x[0]), int(x[1]), int(x[2]), int(x[3])), reader)

  print(tuple(h))
# -> (house(index=1, square_footage=2222, year=1981, list_price=250000),
#     house(index=2, square_footage=1628, year=2009, list_price=185000),
#     ...


(house(index=1, square_footage=2222, year=1981, list_price=250000), house(index=2, square_footage=1628, year=2009, list_price=185000), house(index=3, square_footage=3824, year=1954, list_price=399000), house(index=4, square_footage=1137, year=1993, list_price=150000), house(index=5, square_footage=3560, year=1973, list_price=315000))


Because this file is small, materializing everything with `tuple(h)` is fine. For a large data set you would not — stream the results somewhere instead (a text or JSON file), keeping the one-line-at-a-time property end to end.

*The `zillow.csv` next to this notebook holds only the five rows shown in the lesson, so counts and totals here will differ from the Codecademy workspace, which has the full file.*


**Exercise** — same pattern on `trees.csv`, whose rows are `index`, `width` (inches), `height` (feet), `volume` (ft³).

1. A `tree` namedtuple with those four fields.
2. Inside the `with` block, a `map` iterator called `mapper` that turns each row into a `tree`. Types matter: `index` and `height` are `int`, `width` and `volume` are `float`.

One difference from the Zillow example: this file has a **header line**, so the starter code calls `next(reader)` once before mapping to throw it away. Without that, `int('index')` would blow up on the first row.


In [27]:
import csv
from collections import namedtuple
from functools import reduce

# Code for Checkpoint 1 goes here.
tree = namedtuple("tree", ["index", "width", "height", "volume"])


with open('trees.csv', newline = '') as csvfile:
  reader = csv.reader(csvfile, delimiter=',', quotechar='|')
  next(reader) # Skip the first line in trees.csv that contains the data labels.
  # Code for Checkpoint 2 goes here.
  mapper = map(lambda x: tree(int(x[0]), float(x[1]), int(x[2]), float(x[3])), reader)
  
  trees = tuple(mapper)
  print(trees)
# -> (tree(index=1, width=8.3, height=70, volume=10.3),
#     tree(index=2, width=8.6, height=65, volume=10.3),
#     ...


(tree(index=1, width=8.3, height=70, volume=10.3), tree(index=2, width=8.6, height=65, volume=10.3), tree(index=3, width=8.8, height=63, volume=10.2), tree(index=4, width=10.5, height=72, volume=16.4), tree(index=5, width=10.7, height=81, volume=18.8))


`next(reader)` is the same iterator mechanism as everything else here — it just advances the bookmark past the header so `map` starts on real data. The `float` vs. `int` casts are what make later arithmetic on `width`/`volume` behave; leaving them as strings would make `reduce` concatenate instead of add.

*`trees.csv` beside this notebook is a five-row stand-in (header + first five rows of the classic tree girth/height/volume dataset), so it runs but won't match the workspace file's full output.*


### Processing Data From a CSV File

Materializing every record with `tuple()` is wasteful on a big file. Since `map` over the reader is itself an iterator, stack the other higher-order functions directly on top of it and only pay for the records you keep. Same inside-out reading as before: the reader feeds `map`, `map` feeds `filter`, `filter` feeds `reduce`.

**Example.** Houses built after 1985, then the priciest of those.

1. **filter** the mapped `house` records on `q[2] > 1985` (slot 2 is `year`).
2. **reduce** the survivors with `x if x.list_price > y.list_price else y` — the accumulator carries a whole record, as in the menu example.

The lesson's two snippets each rebuild `h` from `reader`; here they are chained once so the cell runs as-is.


In [28]:
import csv
from collections import namedtuple
from functools import reduce

house = namedtuple("house", ["index", "square_footage", "year", "list_price"])

with open('zillow.csv', newline = '') as csvfile:
  reader = csv.reader(csvfile, delimiter=',', quotechar='|')

  # filter wrapped around map wrapped around the reader
  h = filter(lambda q: q[2] > 1985, map(lambda x: house(int(x[0]), int(x[1]), int(x[2]), int(x[3])), reader))

  houses = tuple(h)
  print(houses) # Print out tuples of houses built after 1985
  # -> (house(index=2, square_footage=1628, year=2009, list_price=185000),
  #     house(index=4, square_footage=1137, year=1993, list_price=150000))

  most_expensive = reduce(lambda x, y: x if x.list_price > y.list_price else y, houses)
  print(most_expensive) # Print out the most expensive house built after 1985
  # -> house(index=2, square_footage=1628, year=2009, list_price=185000)


(house(index=2, square_footage=1628, year=2009, list_price=185000), house(index=4, square_footage=1137, year=1993, list_price=150000))
house(index=2, square_footage=1628, year=2009, list_price=185000)


Two things to notice:

- **The reader is one-shot.** If you ran the lesson's two snippets back to back inside one `with` block, the second `filter(..., map(..., reader))` would see an already-exhausted reader and produce nothing. That is why `houses` is materialized once and the `reduce` runs on the tuple, not on a fresh chain.
- `q[2]` and `q.year` are the same field. Positional access works on a namedtuple, but the named form says what it means; the reduce lambda uses `.list_price` for that reason.

Real-world caveat from the lesson: CSVs are often not clean. Entries can carry leading or trailing whitespace. `int()` and `float()` happen to tolerate it (`int(' 1981')` is fine), but string fields do not — `' Appetizer' == 'Appetizer'` is `False`, so a filter on a text column silently drops rows. Strip with `x[i].strip()` before comparing or storing text.


**Exercise** — same chain on `trees.csv`. The starter now strips the index field (`x[0].strip()`) before casting, putting the whitespace warning into practice.

1. A `filter` iterator `t` over `mapper` keeping trees with `height > 75`, materialized as `trees = tuple(t)`.
2. `reduce` over `trees` to find the **widest** tree — the accumulator keeps whichever record has the larger `width`.


In [29]:
import csv
from collections import namedtuple
from functools import reduce

tree = namedtuple("tree", ["index", "width", "height", "volume"]) 

with open('trees.csv', newline = '') as csvfile:
  reader = csv.reader(csvfile, delimiter=',', quotechar='|')
  next(reader)
  mapper = map(lambda x: tree(int(x[0].strip()), float(x[1]), int(x[2]), float(x[3])), reader)
  
  # Checkpoint 1 code goes here.
  t = filter(lambda x: x.height > 75, mapper)
  trees = tuple(t)

  # Checkpoint 2 code goes here.
  widest = reduce(lambda x, y: x if x.width > y.width else y, trees)
  print(widest)
  # -> tree(index=5, width=10.7, height=81, volume=18.8)   (five-row stand-in file)


tree(index=5, width=10.7, height=81, volume=18.8)


Three iterators stacked inside one `with` block — `reader` → `mapper` → `t` — and nothing is read from disk until `tuple(t)` starts pulling. The filter uses `x.height` rather than `x[2]`, which reads better than the lesson's positional version and does the same thing.

*On the five-row stand-in only tree 5 is taller than 75 feet, so `reduce` gets a one-element tuple and simply returns it. The workspace file has more rows and a different answer.*


### Processing Data From a JSON File

**JSON** is the standard format for data sent over the Internet; API calls almost always return it. The same higher-order functions work on it, with one structural difference from CSV:

| | CSV | JSON |
| --- | --- | --- |
| How it's read | `csv.reader` — a lazy **iterator**, one line at a time | `json.load` — the **whole file** at once, into a dict |
| What you get | list of strings per row | nested dicts/lists mirroring the JSON |
| Memory | scales to huge files | entire file lives in RAM |

Python's `json` library has no streaming reader, so you work with the `dict` it hands back. Nested JSON objects become nested dicts; JSON arrays become lists.

**Example.** `cities.json` has one top-level key, `"cities"`, mapping to a list of city objects:

```
{ "cities": [ { "name": ..., "country": ..., "coordinates": [lat, lon], "continent": ... }, ... ] }
```

Three steps:

1. Open and `json.load` — the file can close after this, since everything is already in `data`.
2. Pull the list out: `cities_list = data["cities"]`, a list of dicts.
3. `map` each dict to a `city` namedtuple by pulling fields **by key**, not by position — `x["name"]` instead of the CSV's `x[0]`. No casting needed either: JSON already carries types, so `coordinates` arrives as a real list of numbers.

The resulting `cities` is an iterable of namedtuples, and `filter`/`reduce` stack on it exactly as they did on the CSV `mapper`.


In [30]:
import json
from collections import namedtuple

city = namedtuple("city", ["name", "country", "coordinates", "continent"])

with open('cities.json') as json_file:
  # load city data into a dictionary with one entry
  data = json.load(json_file) 

# Extract the collection of cities from the dictionary — this is a list of dictionaries
cities_list = data["cities"]

cities = map(lambda x: city(x["name"], x["country"], x["coordinates"], x["continent"]), cities_list)

print(tuple(cities))
# -> (city(name='Tokyo', country='Japan', coordinates=[35.68, 139.69], continent='Asia'),
#     city(name='Paris', country='France', coordinates=[48.86, 2.35], continent='Europe'),
#     ...


(city(name='Tokyo', country='Japan', coordinates=[35.68, 139.69], continent='Asia'), city(name='Mumbai', country='India', coordinates=[19.08, 72.88], continent='Asia'), city(name='Bangkok', country='Thailand', coordinates=[13.76, 100.5], continent='Asia'), city(name='Paris', country='France', coordinates=[48.86, 2.35], continent='Europe'), city(name='Dakar', country='Senegal', coordinates=[14.72, -17.47], continent='Africa'), city(name='New York', country='United States', coordinates=[40.71, -74.01], continent='North America'), city(name='Sydney', country='Australia', coordinates=[-33.87, 151.21], continent='Oceania'))


Notice the `map` sits *outside* the `with` block this time and that is fine — unlike the CSV reader, `data` is a plain dict that outlives the file handle. The laziness is still there, though: `cities` is a `map` iterator and is one-shot, so `tuple(...)` it once if you need it more than once.

*`cities.json` beside this notebook is a seven-city stand-in with the lesson's structure; the workspace file has different entries.*


**Exercise** — filter and reduce on the JSON-sourced `cities`.

1. `asia`: a tuple of every city whose `.continent` is `"Asia"`, via `filter`.
2. `west`: the **westernmost** city in `asia`, i.e. the one with the smallest longitude, `coordinates[1]`. Same accumulator shape as `widest`/`most_expensive`, with `<` on the longitude.

Note the workspace file's top-level key is `"city"`, not the `"cities"` the lesson text shows — the starter code reads `data["city"]`.


In [31]:
import json
from collections import namedtuple
from functools import reduce

city = namedtuple("city", ["name", "country", "coordinates", "continent"])

with open('cities.json') as json_file:
  data = json.load(json_file) 

cities = map(lambda x: city(x["name"], x["country"], x["coordinates"], x["continent"]), data["city"])

# Code for Checkpoint 1 goes here.
asia = tuple(filter(lambda q: q.continent == "Asia", cities))
print(asia)
# -> (city(name='Tokyo', ...), city(name='Mumbai', ...), city(name='Bangkok', ...))   stand-in file

west = None

# Code for Checkpoint 2 goes here.
west = reduce(lambda x, y: x if x.coordinates[1] < y.coordinates[1] else y, asia)
print(west)
# -> city(name='Mumbai', country='India', coordinates=[19.08, 72.88], continent='Asia')   stand-in file


(city(name='Tokyo', country='Japan', coordinates=[35.68, 139.69], continent='Asia'), city(name='Mumbai', country='India', coordinates=[19.08, 72.88], continent='Asia'), city(name='Bangkok', country='Thailand', coordinates=[13.76, 100.5], continent='Asia'))
city(name='Mumbai', country='India', coordinates=[19.08, 72.88], continent='Asia')


`asia` is materialized with `tuple(...)` before the `reduce` because `filter` is one-shot and the tuple is needed twice: once to print, once to reduce. The reduce compares `coordinates[1]` (longitude) but carries and returns the *whole* `city` record, so `west` prints as a full namedtuple rather than a bare number.

*The stand-in `cities.json` beside this notebook carries the list under both `"cities"` and `"city"` so the lesson cell and this checkpoint both run; the workspace file has different entries and a different answer.*


## Q & A (captured as I go)

*Questions posed during the lesson + answers, folded in per sublesson.*

**Q: What does the modulo operator do again?**

`%` returns the **remainder** after division: `17 % 5` → `2` (since $17 = 3 \times 5 + 2$), `10 % 2` → `0`, `9 % 2` → `1`.

Two everyday uses:

- **Divisibility / even-odd test** — `n % 2 == 0` is "n is even"; `n % 3 == 0` is "divisible by 3". This is the usual shape of a predicate handed to `filter` or a lambda.
- **Wrapping an index into a range** — `i % len(items)` cycles 0, 1, 2, 0, 1, 2... however large `i` grows.

Two gotchas: the result takes the sign of the **right** operand, so `-7 % 3` is `2` in Python (C and Java give `-1`), and it works on floats — `7.5 % 2` is `1.5`.

**Q: Quick teaching reference — `map()`, `filter()`, `reduce()`?**

All three take a **function + an iterable** and hand the function each element for you, so the loop disappears. The difference is what the function must *return* and what you get *back*:

| | Function you pass | Returns | Length of result | Import |
|---|---|---|---|---|
| `map(f, it)` | transform: `x -> new value` | lazy iterator | same as input | built-in |
| `filter(pred, it)` | predicate: `x -> True/False` | lazy iterator | $\le$ input | built-in |
| `reduce(acc, it)` | accumulator: `(running, x) -> running` | one value | 1 | `from functools import reduce` |

```python
from functools import reduce
nums = (1, 2, 3, 4, 5)

tuple(map(lambda x: x * x, nums))          # -> (1, 4, 9, 16, 25)   change every element
tuple(filter(lambda x: x % 2 == 0, nums))  # -> (2, 4)              keep some elements
reduce(lambda x, y: x + y, nums)           # -> 15                  collapse to one value
```

How to pick: **map** when the output has the same shape but different values; **filter** when the shape shrinks but values are untouched; **reduce** when many values become one (sum, max, product, joined string).

Reading `reduce`: the lambda's first parameter is the **running result**, the second is the **next element**. `reduce(f, (a, b, c, d))` is `f(f(f(a, b), c), d)`. An optional third argument seeds the running value: `reduce(lambda x, y: x + y, nums, 100)` → `115`, and it also keeps `reduce` from crashing on an empty iterable.

Gotchas:

- `map` and `filter` are **lazy** — they return iterators, not lists. Wrap in `tuple(...)`/`list(...)` to see the values, and an iterator is spent after one pass.
- `filter(None, it)` drops falsy values (`0`, `""`, `None`) with no lambda at all.
- Python's idiom for `map`/`filter` is usually a comprehension: `[x*x for x in nums if x % 2 == 0]` does both in one readable line. `reduce` has no comprehension equivalent, but `sum`, `max`, `min`, `any`, `all` cover the common cases.

### Q: What is an iterator again? (9/22/26)

An **iterator** is an object that hands you one item at a time on request and remembers where it is.

- It has a `__next__` method; `next(it)` gives the next item.
- When it runs out, `next(it)` raises `StopIteration`. A `for` loop catches that and stops.
- It is **stateful and one-shot** — forward only, and once exhausted it stays empty. Make a new one to start over.

An **iterable** is a container you can *get* an iterator from (tuple, list, dict, string, open file) via `iter(...)`. A `for` loop does `iter()` and repeated `next()` for you.

```python
nums = (10, 20, 30)   # iterable: holds all the data
it = iter(nums)       # iterator: a cursor over it
next(it)              # -> 10
next(it)              # -> 20
next(it)              # -> 30
next(it)              # -> StopIteration
```

Why it keeps coming up in this lesson:

- `csv.reader` is an iterator over the file — one line per `next()`, so million-line files never sit fully in RAM.
- `map` and `filter` return iterators, not tuples. Nothing is computed until something pulls from them, hence the `tuple(...)` wrappers and the empty second pass if you iterate twice.
- `map` over `reader` chains two iterators: each `next()` on the map pulls one line from the reader, casts it, and yields one `house`. The file flows through one record at a time.

### Q: What is a `__next__` method? (9/22/26)

The method Python calls when you write `next(obj)`. It is a **dunder** method like `__init__`: you define it on a class, Python calls it behind a built-in.

| You write | Python calls |
| --- | --- |
| `next(it)` | `it.__next__()` |
| `iter(obj)` | `obj.__iter__()` |
| `len(obj)` | `obj.__len__()` |

The **iterator protocol** is just two methods: `__next__` returns the next item or raises `StopIteration`, and `__iter__` returns `self`. A hand-rolled countdown:

```python
class Countdown:
  def __init__(self, start):
    self.n = start
  def __iter__(self):
    return self
  def __next__(self):
    if self.n <= 0:
      raise StopIteration
    self.n -= 1
    return self.n + 1

c = Countdown(3)
next(c)          # -> 3
next(c)          # -> 2
for x in c:      # keeps calling __next__ until StopIteration
  print(x)       # -> 1
```

State lives in `self.n`, which is why iterators are one-shot. `csv.reader`, `map`, and `filter` are objects built this way in C: "`map` calls `next(reader)`" means the map object's `__next__` calls the reader's `__next__`, applies the lambda, and returns the result.

### Q: `__init__` vs. `__iter__` vs. `__next__`? (9/22/26)

All three are dunder methods: you define them, Python calls them at a set moment.

| Method | Python calls it when... | Its job |
| --- | --- | --- |
| `__init__(self, ...)` | you create an object: `Countdown(3)` | set starting state (`self.n = 3`) |
| `__iter__(self)` | `iter(obj)` or the start of a `for` loop | return the iterator to use — for an iterator, `self` |
| `__next__(self)` | `next(obj)` or each pass of a `for` loop | return the next item, or raise `StopIteration` |

```python
c = Countdown(3)   # __init__ runs: self.n = 3
for x in c:        # __iter__ runs once, returns c
  print(x)         # __next__ runs each pass: 3, 2, 1, then StopIteration
```

- `__init__` is the constructor from the OOP lessons; nothing to do with iteration.
- `__iter__` answers "how do I loop over you?" A tuple returns a *fresh* cursor each time (re-loopable); an iterator returns *itself* (one-shot).
- `__next__` does the stepping. Only iterators have it — `next((1, 2, 3))` is a `TypeError`.

`__init__` builds the object, `__iter__` hands out the cursor, `__next__` advances it.

### Q: What is a reader object? (9/22/26)

**The mechanism that actually reads the file** — `open()` only gets the file ready; the reader does the reading, one line per request. Think of it as a **bookmark in a file.** `csv.reader(csvfile)` returns an object that holds no data yet; it just knows which file it points at and that it is sitting at line 1.

Every time you ask for the next line, it reads one line, chops it at the commas, hands back a list of the pieces, and moves the bookmark down one line.

```python
reader = csv.reader(csvfile)
next(reader)   # -> ['1', '2222', '1981', '250000']   line 1, bookmark now on line 2
next(reader)   # -> ['2', '1628', '2009', '185000']   line 2, bookmark now on line 3
```

- The pieces are always **text**, even the numbers — `int('2222')` turns one into a number.
- It reads **one line at a time, only when asked**, so a huge file never fills memory.
- It only works **while the file is open** — keep code that uses it inside the `with` block.

A `for` loop or `map` just keeps asking "next line?" until the file runs out. (`delimiter=','` sets the separator; `quotechar='|'` sets the wrap character for fields containing a comma — harmless here.)

### Q: What is a namedtuple again? (9/22/26)

A **tuple where each slot has a name**, so you pull values out by name instead of by position.

```python
scott = ("Scott", 28, 'A')
scott[1]        # -> 28   ...but is slot 1 the age or the grade?

from collections import namedtuple
student = namedtuple("student", ["name", "age", "grade"])   # define the type
scott = student("Scott", 28, 'A')                            # make one
scott.age       # -> 28
scott[1]        # -> 28   position still works
```

- **Still a tuple** — immutable, so `scott.age = 29` is an error. That is the point: records that can't change underneath you.
- **The definition line is a factory call.** `namedtuple("student", [...])` returns a class; calling the class builds records. Convention: variable name matches the string.
- **Prints self-describing:** `student(name='Scott', age=28, grade='A')`.
- **Use it whenever data has a fixed shape** and you'd otherwise write `x[0]`, `x[1]`, `x[2]` — exactly the CSV case, where `house(int(x[0]), int(x[1]), ...)` turns a list of strings into a record with real field names.

### Q: What does `.strip()` do? (9/23/26)

Removes whitespace (spaces, tabs, newlines) from **both ends** of a string; the middle is untouched.

```python
'  Appetizer  '.strip()      # -> 'Appetizer'
' 1981\n'.strip()            # -> '1981'
'Mixed Green Salad'.strip()  # -> 'Mixed Green Salad'   inner space stays
'$9.95'.strip('$')           # -> '9.95'   strip specific characters instead
```

- `.lstrip()` / `.rstrip()` do one side only.
- Returns a **new** string (strings are immutable), so use the result: `name = x[0].strip()`.

Why it matters for CSVs: `1, Sizzling Canadian Bacon, Appetizer, 9.95` splits into `['1', ' Sizzling Canadian Bacon', ' Appetizer', ' 9.95']`. Numbers survive `int()`/`float()`, but `' Appetizer' == 'Appetizer'` is `False`, so a filter on that column drops every row. Strip inside the map lambda: `menu_item(x[0].strip(), x[1].strip(), float(x[2]))`.

### Q: Arguments for `filter`, `map`, `reduce` again? (9/23/26)

All three take the **function first, iterable second**. The difference is the shape of the function.

| Call | The function must... | Called with | Gives back |
| --- | --- | --- | --- |
| `filter(pred, it)` | take **1** arg, return True/False | each element | the elements that returned True |
| `map(f, it)` | take **1** arg, return a new value | each element | one new value per element |
| `reduce(acc, it)` | take **2** args, return one value | running result, next element | a single final value |

```python
nums = (1, 2, 3, 4)
filter(lambda x: x % 2 == 0, nums)   # -> (2, 4)
map(lambda x: x * 10, nums)          # -> (10, 20, 30, 40)
reduce(lambda x, y: x + y, nums)     # -> 10   runs as ((1 + 2) + 3) + 4
```

The `reduce` lambda's two parameters are **not** two elements: `x` is the result so far, `y` is the next element, and each answer is fed back in as the new `x`. Hence inputs and output must be the same type — in the house example both are whole records and the lambda returns one of them.

The iterable can be anything iterable, including a reader or another `map`/`filter`. `reduce` takes an optional third argument as the starting value: `reduce(lambda x, y: x + y, nums, 100)` → `110`.


## TL;DR

- **Functional style** = pure functions on **immutable** data, written declaratively (say what, not how). Immutability buys thread-safety and no accidental mutation.
- **`namedtuple`** is an immutable tuple with named slots. Define the type once (`house = namedtuple("house", [...])`), then build records; access by `.field` instead of `[i]`. Use it for anything with a fixed shape.
- **`lambda`** is a one-expression anonymous function. Its job is to be passed as an argument, inline or via a variable. Anything longer stays a `def`.
- **The three higher-order functions** take the function first, iterable second:

  | | function shape | result |
  | --- | --- | --- |
  | `filter(pred, it)` | 1 arg → bool | the elements that pass |
  | `map(f, it)` | 1 arg → new value | every element transformed |
  | `reduce(acc, it)` | 2 args (running result, next) → one value | a single value |

  `reduce` lives in `functools`. `map`/`filter` return lazy, **one-shot** iterators — `tuple(...)` them to see or reuse the result.
- **Nesting reads inside-out.** The innermost call runs first. Order carries meaning: filter *before* map when the condition is on raw values, *after* when it's on transformed ones. The `reduce` accumulator must take and return the same type, so map non-numeric data (dicts of tuples, records) to a comparable form first — or carry whole records and compare a field (`x if x.price < y.price else y`).
- **CSV:** `csv.reader` is a lazy iterator (a bookmark in the file), one line per `next()`, every field a **string**. `map` a casting lambda over it inside the `with` block; stack `filter`/`reduce` on top so a huge file never sits in RAM. Skip a header with `next(reader)`; `.strip()` text fields before comparing.
- **JSON:** `json.load` reads the **whole file into a dict** (no streaming). Pull the list out by key, `map` each dict to a namedtuple by key (`x["name"]`) — no casting, JSON already carries types — and the same `filter`/`reduce` chains apply.
- **Iterator mental model:** an object with `__next__` (next item or `StopIteration`) and `__iter__` (returns itself). Containers hand out a fresh cursor each loop; iterators *are* the cursor, so they're spent after one pass.
